# Get Quasar Cadences From Actual Surveys

Let's get us some quasar observing patterns. Build three quasar target lists from Milliquas (VizieR VII/294).

Footprints
----------
* ZTF   : Dec > -28   (survey edge ~-31; margin for field coverage)
* CRTS  : -75 < Dec < 70, |b| > 15
* LSST  : derived from the OpSim baseline v5.3.0 pointing history (see below).
        Requires baseline_v5.3.0_10yrs.db. If absent, script FAILS rather than
        substituting an approximate cut -- set ALLOW_APPROX_LSST = True to
        override explicitly.

Magnitude limits (Milliquas Rmag)
---------------------------------
* CRTS  : 12.0 - 19.5   (saturation to practical faint limit)
* ZTF   : 13.0 - 20.3
* LSST  : 16.0 - 23.0   (single-visit; coadd goes deeper)



In [1]:
import json
import sqlite3
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd
from astropy import units as u
from astropy.coordinates import SkyCoord



Some configs:

In [2]:
# ----------------------------------------------------------------------------
# Configuration
# ----------------------------------------------------------------------------

SEED = 42
OUTDIR = Path("/Users/danielahuppenkothen/work/data/quasar_cadences/")

OPSIM_DB = "/Users/danielahuppenkothen/work/data/quasar_cadences/baseline_v5.3.0_10yrs.db"
ALLOW_APPROX_LSST = False        # set True to proceed without the OpSim file

N_PER_SURVEY = 10000

# Footprint definitions
ZTF_DEC_MIN = -28.0
CRTS_DEC_MIN, CRTS_DEC_MAX = -75.0, 70.0
GAL_LAT_MIN = 15.0               # |b| cut, applied to all lists

# LSST footprint derivation from OpSim
LSST_NSIDE = 64                  # ~0.9 deg pixels
LSST_MIN_VISITS = 100            # pixels with fewer visits are not "in footprint"
FOV_RADIUS_DEG = 1.75            # circular approximation (deliberate)

# Magnitude limits (Milliquas Rmag)
CRTS_BRIGHT, CRTS_FAINT = 12.0, 19.5
ZTF_BRIGHT, ZTF_FAINT = 13.0, 20.3
LSST_BRIGHT, LSST_FAINT = 16.0, 23.0

DEC_BIN_WIDTH = 15.0             # for stratified sampling

# ----------------------------------------------------------------------------


Useful functions:

In [3]:
def fetch_milliquas():
    """Pull the Million Quasars catalog from VizieR."""
    from astroquery.vizier import Vizier

    print("Querying VizieR VII/294 (Milliquas)...")
    v = Vizier(
        columns=["Name", "RAJ2000", "DEJ2000", "z", "Rmag", "Bmag", "Type"],
        row_limit=-1,
    )
    cats = v.get_catalogs("VII/294")
    if len(cats) == 0:
        raise RuntimeError("VizieR returned no catalogs for VII/294.")
    df = cats[0].to_pandas()

    expected = {"Name", "RAJ2000", "DEJ2000", "z", "Rmag", "Type"}
    missing = expected - set(df.columns)
    if missing:
        raise KeyError(
            f"VizieR columns changed. Missing {missing}. Got: {list(df.columns)}"
        )

    df = df[df["Type"].astype(str).str.startswith("Q")]
    df = df.dropna(subset=["RAJ2000", "DEJ2000", "Rmag"])

    c = SkyCoord(ra=df["RAJ2000"].values * u.deg,
                 dec=df["DEJ2000"].values * u.deg, frame="icrs")
    df = df.assign(glat=c.galactic.b.deg)

    df = df[df["glat"].abs() > GAL_LAT_MIN]
    print(f"  {len(df)} type-Q quasars with |b| > {GAL_LAT_MIN}")
    return df.reset_index(drop=True)


In [4]:
def lsst_footprint_mask(ra, dec):
    """
    Boolean mask: which (ra, dec) fall in the LSST footprint, derived from the
    OpSim v5.3.0 visit history.

    Builds a HEALPix visit-count map from pointing centers, smoothed by the
    field of view, then requires >= LSST_MIN_VISITS at each target position.
    """
    import healpy as hp

    if not Path(OPSIM_DB).exists():
        if not ALLOW_APPROX_LSST:
            raise FileNotFoundError(
                f"{OPSIM_DB} not found. LSST footprint requires the OpSim database.\n"
                "Download it, or set ALLOW_APPROX_LSST = True to use a crude "
                "Dec < +12 cut instead (documented as an approximation)."
            )
        print("  WARNING: using approximate LSST cut (Dec < +12), no OpSim data.")
        return dec < 12.0

    print(f"  Reading {OPSIM_DB} ...")
    con = sqlite3.connect(OPSIM_DB)

    cols = pd.read_sql_query("PRAGMA table_info(observations)", con)
    names = set(cols["name"])
    ra_col = next((c for c in ("fieldRA", "fieldra", "ra") if c in names), None)
    dec_col = next((c for c in ("fieldDec", "fielddec", "dec") if c in names), None)
    if ra_col is None or dec_col is None:
        con.close()
        raise KeyError(
            f"Could not find pointing columns in OpSim schema.\n"
            f"Available: {sorted(names)}"
        )
    print(f"  Using pointing columns: {ra_col}, {dec_col}")

    # Subsample visits: the footprint shape converges long before 2M rows.
    ptg = pd.read_sql_query(
        f"SELECT {ra_col} AS ra, {dec_col} AS dec FROM observations", con
    )
    con.close()
    print(f"  {len(ptg)} visits")

    npix = 12 * LSST_NSIDE ** 2
    counts = np.zeros(npix, dtype=np.int32)

    vec = hp.ang2vec(ptg["ra"].to_numpy(), ptg["dec"].to_numpy(), lonlat=True)
    radius = np.radians(FOV_RADIUS_DEG)
    # Accumulate in chunks to bound memory.
    for i in range(0, len(vec), 20000):
        for v in vec[i:i + 20000]:
            counts[hp.query_disc(LSST_NSIDE, v, radius, inclusive=False)] += 1

    good = counts >= LSST_MIN_VISITS
    frac = good.sum() / npix
    print(f"  Footprint: {good.sum()} pixels ({frac * 100:.1f}% of sky), "
          f">= {LSST_MIN_VISITS} visits")

    pix = hp.ang2pix(LSST_NSIDE, ra, dec, lonlat=True)
    return good[pix]


In [5]:
def stratified_sample(df, n, rng):
    """Sample n rows, spread evenly across declination bins."""
    if n is None or len(df) <= n:
        return df.copy()

    edges = np.arange(np.floor(df["DEJ2000"].min() / DEC_BIN_WIDTH) * DEC_BIN_WIDTH,
                      df["DEJ2000"].max() + DEC_BIN_WIDTH, DEC_BIN_WIDTH)
    bins = pd.cut(df["DEJ2000"], bins=edges)
    n_bins = bins.nunique()
    if n_bins == 0:
        return df.sample(n, random_state=int(rng.integers(2**31)))

    per_bin = max(1, n // n_bins)
    out = (df.groupby(bins, observed=True, group_keys=False)
             .apply(lambda g: g.sample(min(len(g), per_bin),
                                       random_state=int(rng.integers(2**31)))))
    # Top up from the remainder if stratification undershot.
    if len(out) < n:
        rest = df.drop(out.index)
        extra = min(n - len(out), len(rest))
        if extra > 0:
            out = pd.concat([out, rest.sample(extra,
                                              random_state=int(rng.integers(2**31)))])
    return out.head(n).copy()


In [6]:
def to_output(df, survey):
    out = pd.DataFrame({
        "object_id": df["Name"].astype(str).str.replace(r"\s+", "_", regex=True),
        "ra": df["RAJ2000"].to_numpy(),
        "dec": df["DEJ2000"].to_numpy(),
        "z": df["z"].to_numpy(),
        "rmag": df["Rmag"].to_numpy(),
        "glat": df["glat"].to_numpy(),
        "survey_list": survey,
    })
    return out.reset_index(drop=True)


First, download the catalogue:

In [13]:
rng = np.random.default_rng(SEED)
mq = fetch_milliquas()


Querying VizieR VII/294 (Milliquas)...
  859025 type-Q quasars with |b| > 15.0


Now we can make a mask for the three instruments based on their positions:

In [17]:
ra = mq["RAJ2000"].to_numpy()
dec = mq["DEJ2000"].to_numpy()

print("\nDeriving footprint masks...")
in_ztf = dec > ZTF_DEC_MIN
in_crts = (dec > CRTS_DEC_MIN) & (dec < CRTS_DEC_MAX)
print("  LSST:")
in_lsst = lsst_footprint_mask(ra, dec)



Deriving footprint masks...
  LSST:
  Reading /Users/danielahuppenkothen/work/data/quasar_cadences/baseline_v5.3.0_10yrs.db ...
  Using pointing columns: fieldRA, fieldDec
  1844571 visits
  Footprint: 32644 pixels (66.4% of sky), >= 100 visits


In [23]:
print(f"  ZTF footprint:  {in_ztf.sum()}")
print(f"  CRTS footprint: {in_crts.sum()}")
print(f"  LSST footprint: {in_lsst.sum()}")


  ZTF footprint:  839928
  CRTS footprint: 857163
  LSST footprint: 319180


And the magnitude cuts:

In [24]:
mag = mq["Rmag"].to_numpy()
ok_crts_mag = (mag > CRTS_BRIGHT) & (mag < CRTS_FAINT)
ok_ztf_mag = (mag > ZTF_BRIGHT) & (mag < ZTF_FAINT)
ok_lsst_mag = (mag > LSST_BRIGHT) & (mag < LSST_FAINT)


We'll make a stratified sample:

In [27]:
lists = {}

# --- per-survey lists -------------------------------------------------
lists["ztf"] = stratified_sample(mq[in_ztf & ok_ztf_mag], N_PER_SURVEY, rng)
lists["crts"] = stratified_sample(mq[in_crts & ok_crts_mag], N_PER_SURVEY, rng)
lists["lsst"] = stratified_sample(mq[in_lsst & ok_lsst_mag], N_PER_SURVEY, rng)



Let's write these to file:

In [29]:
print("\nWriting lists:")
summary = {}
for name, df in lists.items():
    out = to_output(df, name)
    path = OUTDIR / f"targets_{name}.csv"
    out.to_csv(path, index=False)

    s = {
        "n": len(out),
        "dec_min": float(out["dec"].min()),
        "dec_max": float(out["dec"].max()),
        "rmag_min": float(out["rmag"].min()),
        "rmag_max": float(out["rmag"].max()),
    }
    summary[name] = s

    print(f"  {path}: {len(out)} targets, "
          f"Dec [{s['dec_min']:.1f}, {s['dec_max']:.1f}], "
          f"R [{s['rmag_min']:.1f}, {s['rmag_max']:.1f}]")

meta = {
    "generated_utc": datetime.now(timezone.utc).isoformat(),
    "seed": SEED,
    "catalog": "VizieR VII/294 (Milliquas)",
    "gal_lat_min": GAL_LAT_MIN,
    "footprints": {
        "ztf": f"Dec > {ZTF_DEC_MIN}",
        "crts": f"{CRTS_DEC_MIN} < Dec < {CRTS_DEC_MAX}",
        "lsst": (f"OpSim {OPSIM_DB}, nside={LSST_NSIDE}, "
                 f"min_visits={LSST_MIN_VISITS}, fov={FOV_RADIUS_DEG}deg"
                 if Path(OPSIM_DB).exists() else "APPROXIMATE Dec < +12"),
    },
    "mag_limits": {
        "crts": [CRTS_BRIGHT, CRTS_FAINT],
        "ztf": [ZTF_BRIGHT, ZTF_FAINT],
        "lsst": [LSST_BRIGHT, LSST_FAINT],
    },
    "summary": summary,
}
(OUTDIR / "targets_metadata.json").write_text(json.dumps(meta, indent=2))
print("\nWrote targets_metadata.json")



Writing lists:
  /Users/danielahuppenkothen/work/data/quasar_cadences/targets_ztf.csv: 10000 targets, Dec [-28.0, 88.7], R [13.6, 20.3]
  /Users/danielahuppenkothen/work/data/quasar_cadences/targets_crts.csv: 10000 targets, Dec [-74.9, 69.9], R [13.0, 19.5]
  /Users/danielahuppenkothen/work/data/quasar_cadences/targets_lsst.csv: 10000 targets, Dec [-87.3, 32.7], R [16.0, 23.0]

Wrote targets_metadata.json


## Fetching ZTF Data

Let's try to fetch some ZTF light curves:

In [7]:

import hats
import lsdb
import re

In [8]:
#!/usr/bin/env python
"""
Crossmatch a Milliquas quasar sample against ZTF DR24 light curves (HATS on S3)
and export flat CSVs for analysis.

Outputs
-------
ztf_out/epochs.csv      One row per photometric epoch (long format).
ztf_out/objects.csv     One row per matched ZTF object (per filter/field/quadrant).
ztf_out/targets_matched.csv
                        One row per input target, with match counts.
ztf_out/metadata.json   Provenance: release, radius, filters, counts, timestamp.

Notes
-----
- ZTF objects are defined per (filter, field, CCD-quadrant), so one astrophysical
  quasar normally maps to SEVERAL ZTF objectids. All are kept; group by filterid.
- Times are HELIOCENTRIC MJD (hmjd), not plain MJD. Kept as-is; see TIME_SYSTEM.
- Requires no credentials. Anonymous S3 reads.

Install:
    pip install "lsdb>=0.8.1" s3fs pyarrow pandas astropy "dask[distributed]"
"""

import json
import os
from datetime import datetime, timezone
from pathlib import Path

import lsdb
import pandas as pd
from dask.distributed import Client



In [24]:
# ----------------------------------------------------------------------------
# Configuration
# ----------------------------------------------------------------------------

TARGETS_CSV = OUTDIR/"targets_ztf.csv"        # needs: object_id, ra, dec  (+ optional z, rmag)
ZTF_OUTDIR = OUTDIR/"ztf_data"
MATCH_RADIUS_ARCSEC = 1.0          # ZTF pixel scale is 1"/px; 2.0 also defensible
MIN_EPOCHS = 100                    # pre-filter on nepochs via Parquet stats
DROP_BAD_CATFLAGS = True           # keep only catflags == 0
TIME_SYSTEM = "hmjd_heliocentric"  # recorded in metadata; do NOT silently mix with MJD

ZTF_BUCKET = "ipac-irsa-ztf"
ZTF_LC_HATS = f"s3://{ZTF_BUCKET}/ztf/enhanced/dr24/lc/hats"
ZTF_MARGIN = f"{ZTF_LC_HATS}/ztf_dr24_lc-hats_margin_10arcsec"
RELEASE = "ZTF_DR24_HATS"

LC_COLUMNS = ["objectid", "objra", "objdec", "filterid", "nepochs", "lightcurve"]
FILTER_NAMES = {1: "g", 2: "r", 3: "i"}

# ----------------------------------------------------------------------------


def load_targets():
    if not Path(TARGETS_CSV).exists():
        raise FileNotFoundError(
            f"{TARGETS_CSV} not found. Expected columns: object_id, ra, dec."
        )
    df = pd.read_csv(TARGETS_CSV)

    required = {"object_id", "ra", "dec"}
    missing = required - set(df.columns)
    if missing:
        raise KeyError(f"{TARGETS_CSV} missing columns: {missing}. Found: {list(df.columns)}")

    bad = df[["ra", "dec"]].isna().any(axis=1)
    if bad.any():
        raise ValueError(f"{bad.sum()} rows have NaN ra/dec. Clean targets.csv first.")

    if not df["ra"].between(0, 360).all() or not df["dec"].between(-90, 90).all():
        raise ValueError("ra/dec outside valid ranges.")

    if df["object_id"].duplicated().any():
        raise ValueError("object_id values are not unique.")

    if N_TARGETS is not None and len(df) > N_TARGETS:
        df = df.sample(N_TARGETS, random_state=42).reset_index(drop=True)
    
    print(f"Loaded {len(df)} targets.")
    return df


def build_catalogs(targets):
    """Milliquas side as a HATS catalog; ZTF side lazily from S3."""
    left = lsdb.from_dataframe(
        targets[["object_id", "ra", "dec"]],
        catalog_name="milliquas_sample",
        ra_column="ra",
        dec_column="dec",
        margin_threshold=None,   # left catalog needs no margin
    )

    row_filters = [["nepochs", ">", MIN_EPOCHS]] if MIN_EPOCHS else None

    right = lsdb.open_catalog(
        ZTF_LC_HATS,
        columns=LC_COLUMNS,
        margin_cache=ZTF_MARGIN,   # required for correct crossmatch at partition edges
        filters=row_filters,
    )
    return left, right


def run_crossmatch(left, right, n_neighbours=12):
    """Plan lazily, then execute under a Dask client."""
    xm = left.crossmatch(
        right,
        radius_arcsec=MATCH_RADIUS_ARCSEC,
        n_neighbors=n_neighbours,                 # keep ALL ZTF objects in the cone, not just nearest
        suffixes=("_mq", "_ztf"),
    )

    n_workers = min(os.cpu_count() or 4, 8)
    with Client(n_workers=n_workers, threads_per_worker=1, memory_limit=None) as client:
        print(f"Dask dashboard: {client.dashboard_link}")
        result = xm.compute()

    print(f"Crossmatch returned {len(result)} ZTF objects.")
    return result


def flatten(result, targets):
    """Explode nested lightcurve column into long-format epochs."""
    if len(result) == 0:
        raise RuntimeError(
            "Crossmatch returned zero rows. Check MATCH_RADIUS_ARCSEC, MIN_EPOCHS, "
            "and that targets fall in ZTF's footprint (Dec > -31)."
        )

    result = result.rename(columns={
        "object_id_mq": "object_id",
        "objectid_ztf": "objectid",
        "filterid_ztf": "filterid",
        "objra_ztf": "objra",
        "objdec_ztf": "objdec",
        "nepochs_ztf": "nepochs",
        "lightcurve_ztf": "lightcurve",
    })
    required = {"object_id", "objectid", "filterid", "lightcurve"}
    missing = required - set(result.columns)
    if missing:
        raise KeyError(f"After rename, missing {missing}. Got: {sorted(result.columns)}")


    obj_rows, epoch_frames = [], []

    for _, row in result.iterrows():
        oid = row["objectid"]
        fid = row["filterid"]
        band = FILTER_NAMES.get(fid, f"fid{fid}")

        lc = row["lightcurve"]
        if lc is None or len(lc) == 0:
            continue
        lc = pd.DataFrame(lc)

        n_raw = len(lc)
        if DROP_BAD_CATFLAGS:
            lc = lc[lc["catflags"] == 0]

        obj_rows.append({
            "object_id": row["object_id"],
            "ztf_objectid": oid,
            "band": band,
            "filterid": fid,
            "objra": row["objra"],
            "objdec": row["objdec"],
            "sep_arcsec": row.get("_dist_arcsec"),
            "n_epochs_raw": n_raw,
            "n_epochs_clean": len(lc),
        })

        if len(lc) == 0:
            continue

        e = pd.DataFrame({
            "object_id": row["object_id"],
            "ztf_objectid": oid,
            "band": band,
            "hmjd": lc["hmjd"].to_numpy(),
            "mag": lc["mag"].to_numpy(),
            "magerr": lc["magerr"].to_numpy(),
            "catflags": lc["catflags"].to_numpy(),
        })
        epoch_frames.append(e)

    objects = pd.DataFrame(obj_rows)
    epochs = (pd.concat(epoch_frames, ignore_index=True)
              if epoch_frames else pd.DataFrame())

    # Dedupe: same object can appear via overlapping fields.
    if len(epochs):
        before = len(epochs)
        epochs = epochs.drop_duplicates(subset=["ztf_objectid", "hmjd"])
        if before != len(epochs):
            print(f"Dropped {before - len(epochs)} duplicate (objectid, hmjd) rows.")
        epochs = epochs.sort_values(["object_id", "band", "hmjd"]).reset_index(drop=True)

    # Per-target rollup, including targets with no match.
    if len(objects):
        agg = objects.groupby("object_id").agg(
            n_ztf_objects=("ztf_objectid", "nunique"),
            n_bands=("band", "nunique"),
            n_epochs_clean=("n_epochs_clean", "sum"),
        ).reset_index()
    else:
        agg = pd.DataFrame(columns=["object_id", "n_ztf_objects", "n_bands", "n_epochs_clean"])

    tm = targets.merge(agg, on="object_id", how="left")
    tm[["n_ztf_objects", "n_bands", "n_epochs_clean"]] = (
        tm[["n_ztf_objects", "n_bands", "n_epochs_clean"]].fillna(0).astype(int)
    )

    return objects, epochs, tm




In [25]:
def write_per_object(epochs, tm, outdir):
    """
    Write one CSV per target into outdir/lightcurves/, plus a master index.
 
    One file per TARGET (not per ZTF objectid): a quasar maps to several ZTF
    objects, so those are kept as rows within the file, distinguished by the
    ztf_objectid and band columns.
    """
    lcdir = outdir / "lightcurves"
    lcdir.mkdir(exist_ok=True)
 
    index_rows = []
    for object_id, grp in epochs.groupby("object_id", sort=True):
        # Milliquas names can contain characters that are awkward in paths.
        safe = re.sub(r"[^A-Za-z0-9_.+-]", "_", str(object_id))
        fname = f"{safe}.csv"
 
        grp = grp.sort_values(["band", "hmjd"])
        grp[["ztf_objectid", "band", "hmjd", "mag", "magerr", "catflags"]].to_csv(
            lcdir / fname, index=False
        )
 
        per_band = grp.groupby("band").size().to_dict()
        index_rows.append({
            "object_id": object_id,
            "file": f"lightcurves/{fname}",
            "n_epochs": len(grp),
            "n_ztf_objects": grp["ztf_objectid"].nunique(),
            "bands": ",".join(sorted(per_band)),
            "n_g": per_band.get("g", 0),
            "n_r": per_band.get("r", 0),
            "n_i": per_band.get("i", 0),
            "hmjd_min": grp["hmjd"].min(),
            "hmjd_max": grp["hmjd"].max(),
            "baseline_days": grp["hmjd"].max() - grp["hmjd"].min(),
            "mean_mag": grp["mag"].mean(),
        })
 
    master = pd.DataFrame(index_rows)
 
    # Left-join onto the full target list so unmatched targets stay visible
    # with matched=False rather than silently vanishing.
    keep = [c for c in ("object_id", "ra", "dec", "z", "rmag") if c in tm.columns]
    master = tm[keep].merge(master, on="object_id", how="left")
    master["file"] = master["file"].fillna("")
    for c in ("n_epochs", "n_ztf_objects", "n_g", "n_r", "n_i"):
        master[c] = master[c].fillna(0).astype(int)
    master["bands"] = master["bands"].fillna("")
    master["matched"] = master["n_epochs"] > 0
 
    master.to_csv(outdir / "master.csv", index=False)
    print(f"Wrote {len(index_rows)} light curve files to {lcdir}/")
    return master
 
 
def report(objects, epochs, tm):
    n_tot = len(tm)
    n_matched = int((tm["n_ztf_objects"] > 0).sum())
    frac_empty = 1 - n_matched / n_tot if n_tot else 1.0
 
    print("\n--- QC ---")
    print(f"targets:            {n_tot}")
    print(f"matched:            {n_matched}  (frac_empty = {frac_empty:.3f})")
    print(f"ZTF objects:        {len(objects)}")
    print(f"clean epochs:       {len(epochs)}")
 
    if n_matched:
        m = tm[tm["n_ztf_objects"] > 0]
        print(f"median ZTF obj/target:   {m['n_ztf_objects'].median():.0f}")
        print(f"median clean epochs/tgt: {m['n_epochs_clean'].median():.0f}")
    if len(epochs):
        print(f"hmjd range:         {epochs['hmjd'].min():.1f} - {epochs['hmjd'].max():.1f}")
        print("\nepochs per band:")
        print(epochs.groupby("band").size().to_string())
 
    if frac_empty > 0.2:
        print(f"\n*** WARNING: frac_empty = {frac_empty:.3f} > 0.2 ***")
        print("Check: targets below Dec -31 are outside ZTF coverage;")
        print("MIN_EPOCHS may be too strict; radius may be too small.")
 
    return {"n_targets": n_tot, "n_matched": n_matched, "frac_empty": frac_empty}

def build_left(targets):
    """Local only. Fast."""
    return lsdb.from_dataframe(
        targets[["object_id", "ra", "dec"]],
        catalog_name="milliquas_sample",
        ra_column="ra",
        dec_column="dec",
        margin_threshold=None,
        should_generate_moc=True,
    )


def build_objects_catalog():
    """S3 metadata read — cache this."""
    return lsdb.open_catalog(
        f"s3://{ZTF_BUCKET}/ztf/enhanced/dr24/objects/hats",
        columns=["oid", "ra", "dec", "filtercode", "ngoodobsrel"],
        margin_cache=f"s3://{ZTF_BUCKET}/ztf/enhanced/dr24/objects/hats/"
                     f"ztf_dr24_objects-hats_margin_10arcsec",
    )

def build_lc_catalog():
    """S3 metadata read — cache this."""
    return lsdb.open_catalog(ZTF_LC_HATS, columns=LC_COLUMNS)



In [26]:
ZTF_OUTDIR.mkdir(exist_ok=True)
N_TARGETS = 550

targets = load_targets()


Loaded 550 targets.


In [27]:
left, right = build_catalogs(targets)
result = run_crossmatch(left, right)
objects, epochs, tm = flatten(result, targets)
stats = report(objects, epochs, tm)
write_per_object(epochs, tm, ZTF_OUTDIR)
objects.to_csv(ZTF_OUTDIR / "objects.csv", index=False)


/Users/danielahuppenkothen/work/sw/miniforge3/envs/periodicity/lib/python3.14/site-packages/lsdb/catalog/catalog.py:410: FutureWarning: The default suffix behavior will change from applying suffixes to all columns to only applying suffixes to overlapping columns in a future release.To maintain the current behavior, explicitly set `suffix_method='all_columns'`. To change to the new behavior, set `suffix_method='overlapping_columns'`.
  warnings.warn(


Dask dashboard: http://127.0.0.1:8787/status


Computing Catalog:   0%|          | 0/471 [00:00<?, ?it/s]

Crossmatch returned 1284 ZTF objects.
Dropped 1 duplicate (objectid, hmjd) rows.

--- QC ---
targets:            550
matched:            444  (frac_empty = 0.193)
ZTF objects:        1284
clean epochs:       594719
median ZTF obj/target:   3
median clean epochs/tgt: 1058
hmjd range:         58197.3 - 60969.5

epochs per band:
band
g    241471
i     48415
r    304833
Wrote 443 light curve files to /Users/danielahuppenkothen/work/data/quasar_cadences/ztf_data/lightcurves/


In [29]:
tm.to_csv(OUTDIR / "targets_matched.csv", index=False)

meta = {
    "release": RELEASE,
    "hats_path": ZTF_LC_HATS,
    "match_radius_arcsec": MATCH_RADIUS_ARCSEC,
    "min_epochs": MIN_EPOCHS,
    "dropped_bad_catflags": DROP_BAD_CATFLAGS,
    "time_system": TIME_SYSTEM,
    "time_column": "hmjd",
    "generated_utc": datetime.now(timezone.utc).isoformat(),
    **stats,
}
(OUTDIR / "metadata.json").write_text(json.dumps(meta, indent=2))

print(f"\nWrote {OUTDIR}/epochs.csv, objects.csv, targets_matched.csv, metadata.json")



Wrote /Users/danielahuppenkothen/work/data/quasar_cadences/epochs.csv, objects.csv, targets_matched.csv, metadata.json


In [31]:
from pathlib import Path
from validate_ztf import load, check_structure, check_photometry, check_cadence
